# 03 — Dựng năm bảng dimension từ Silver

### 1. Xem khóa và ngày nhận

Bắt đầu từ `delivery_silver`. Notebook gọi lần lượt hai hàm dimension của Prefect.

In [ ]:
import sys

sys.path.insert(0, "/home/jovyan/pipeline")
import steps

spark = steps.get_spark("Delivery - Dimensions")

In [ ]:
silver = spark.read.format("delta").load(steps.SILVER)
silver.select("courier_id", "region_id", "aoi_id", "aoi_type", "accept_ts").show(5, truncate=False)

### 2. Dựng lịch sử shipper SCD2

Với mỗi shipper và ngày nhận, chọn vùng có nhiều đơn nhất. Khi bằng số
đơn, chọn `region_id` nhỏ hơn. Mỗi lần vùng đổi tạo một dòng lịch sử mới.
Fact dùng ngày nhận để ghép đúng khoảng `[effective_date, end_date)`.

In [1]:
result = steps.build_dim_courier(spark)
result

{'dim_rows': 226, 'couriers': 203}

In [ ]:
spark.read.format("delta").load(steps.DIM).select(
    "courier_id", "region_id", "effective_date", "end_date", "is_active",
).show(5, truncate=False)

### 3. Dựng vùng, AOI, ngày và giờ

Vùng và AOI lấy khóa duy nhất từ Silver. Tọa độ hợp lệ dùng để tính tâm.
Dimension ngày phủ khoảng ngày nhận và giao. Dimension giờ có 24 dòng.

In [1]:
result = steps.build_star_dimensions(spark)
result

{'dim_region': 17, 'dim_aoi': 8152, 'dim_date': 189, 'dim_hour': 24}